# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/train.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Dataset FF++](#2-dataset-faceforensics)
3. [Funções auxiliares](#3-funções-auxiliares)
4. [Treino](#4-treino)

## 1. Configuração

In [ ]:
# @title instala dependências

%pip install -q tensorflow keras-hub

In [ ]:
# @title Imports

import os, random
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_hub

In [ ]:
# @title Configurações

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-4
FEATURE_DIM = 768
DATASET_ROOT = "/dataset/FaceForensics"
FEATURES_CACHE = "/dataset/siglip2_features_keras.npz"
AUTOTUNE = tf.data.AUTOTUNE

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponível: {bool(tf.config.list_physical_devices("GPU"))}')

## 2. Dataset FaceForensics++

No arquivo **face_extraction.ipynb** foi realizada a extração dos rostos frame por frame nos vídeos do dataset. Para melhorar a otimização, foram utilizados os seguintes filtros na extração:

- SKIP_FRAMES = 9     
_0 = todos; 9 = pula 9 (processa 1 a cada 10)_

- MAX_FRAMES_PER_VIDEO = 30          
_None = todos; ex: 100 = máx 100 frames que contenham rostos são salvos_

- MAX_READ_LIMIT = 1000                 
_None = todos; ex: 1000 = máx 1000 frames lidos por vídeo_

- MAX_VIDEOS = None                 
_None = todos; ex: 10 = máx 10 vídeos processados_

XXX imagens coloridas 200×250 divididas em 2 classes.  
Split padrão: **720 treino / 140 validação / 140 teste**.

| Índice | Classe 
|---|---
| 0 | real 
| 1 | fake

Há também o pré-processamento para o siglip2 que é o que será usado no modelo, as imagens são salvas em 224x224 (float 32)

## 3. Funções auxiliares

In [ ]:
# @title Carrega Siglip2 sem a cabeça classificadora
input_data = {
  "images": np.ones(shape=(1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32),
}

backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
backbone.trainable = False
out = backbone(input_data)

print(f"Keys in output: {out.keys()}")
print(f"Image embedding: {out['image_embedding'].shape}")


In [ ]:
# @title Carrega imagens pré-processadas para o Siglip2

def collect_dataset(root):
    root = Path(root)
    paths, labels, vids = [], [], []
    for class_name, label in [("real", 0), ("fake", 1)]:
        class_dir = root / class_name
        if not class_dir.exists():
                continue
        for npy_file in class_dir.rglob("*_siglip2.npy"):
            paths.append(str(npy_file))
            labels.append(label)
            vids.append(npy_file.parent.name)
    return paths, np.array(labels, np.int32), np.array(vids)

npy_paths, labels, video_ids = collect_dataset(DATASET_ROOT)

print(f"Total: {len(npy_paths)} | Real: {sum(labels==0)} | Fake: {sum(labels==1)}")
print(f"Videos unicos: {len(set(video_ids))}")

real_vids = set(v for v, l in zip(video_ids, labels) if l == 0)
fake_vids = set(v for v, l in zip(video_ids, labels) if l == 1)
colisao = real_vids & fake_vids

if colisao:
    print(f"\n⚠️ {len(colisao)} videos com mesmo nome em real e fake!")
    video_ids = np.array([f"{'real' if l == 0 else 'fake'}_{v}" 
                          for v, l in zip(video_ids, labels)])
else:
    print("\n✓ Sem colisao")

for vid, c in Counter(video_ids).most_common(10):
    print(f"  {vid}: {c} frames")

In [ ]:
# @title Extrai features do Siglip2 e salva em cache .npz

def extract_all(paths, backbone, bs=32):
    feats = []
    for i in range(0, len(paths), bs):
        batch = np.stack([np.load(p) for p in paths[i:i+bs]])
        emb = backbone({"images": batch})["image_embedding"].numpy()
        feats.append(emb)
        if i % 1000 == 0: print(f"{i}/{len(paths)}")
    return np.concatenate(feats, axis=0)

if Path(FEATURES_CACHE).exists():
    d = np.load(FEATURES_CACHE)
    features, labels, video_ids = d["features"], d["labels"], d["video_ids"]
    print("Cache carregado")
else:
    features = extract_all(npy_paths, backbone)
    np.savez(FEATURES_CACHE, features=features, labels=labels, video_ids=video_ids)
    print("Features extraidas e salvas")

print(f"Features: {features.shape}")

## 4. Treino

In [ ]:
# @title Split dataset em treino/validação/teste por vídeo
def split_by_video(features, labels, vids, train=0.7, val=0.2, seed=42):
    random.seed(seed)
    vid_to_idx = defaultdict(list)
    for i, v in enumerate(vids):
        vid_to_idx[v].append(i)
    
    all_vids = list(vid_to_idx.keys())
    random.shuffle(all_vids)
    
    n = len(all_vids)
    n_train = int(n * train)
    n_val = int(n * val)

    train_v = all_vids[:n_train]
    val_v = all_vids[n_train:n_train + n_val]
    test_v = all_vids[n_train + n_val:]
    
    def idxs(vlist):
        return [i for v in vlist for i in vid_to_idx[v]]
    
    return (features[idxs(train_v)], labels[idxs(train_v)],
            features[idxs(val_v)], labels[idxs(val_v)],
            features[idxs(test_v)], labels[idxs(test_v)],
            test_v)

X_train, y_train, X_val, y_val, X_test, y_test, test_vids = split_by_video(features, labels, video_ids)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


In [ ]:
# @title Cria datasets de treino, validação e teste

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [ ]:
# @title Constrói o modelo completo
model = keras.Sequential([
    keras.layers.Input(shape=(FEATURE_DIM,)),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=keras.optimizers.Adam(LR), loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.AUC(name='auc')])
model.summary()


In [ ]:
# @title Callbacks

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint(filepath='model.{epoch:02d}-{val_loss:.2f}.h5', monitor='val_auc', save_best_only=True, mode='max')
]

In [ ]:
# @title Treinamento

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

print(f'\nAcurácia de validação (Fase 1): {history.history["val_accuracy"][-1]:.1%}')